# Schätzung eines Nachfragesystems für Haushaltsenergie mit scheinbar unverbundener Regression

## Zusammenfassung

Ein regionales Versorgungsunternehmen schätzt ein **Energienachfragesystem** für Haushalte mit zwei Gleichungen — die Budgetanteile von Strom und Erdgas als Funktionen des Eigenpreises, des Kreuzpreises und des Haushaltseinkommens — mit **PROC SYSLIN** und der Methode der scheinbar unverbundenen Regression (SUR). Die beiden Anteilsgleichungen teilen sich ein gemeinsames Haushaltsbudget, sodass ihre Fehler kreuzkorreliert sind; SUR schätzt die Gleichungen gemeinsam, um diese Korrelation auszunutzen, gewinnt vorzeichenkohärente Eigen- und Kreuzpreiseffekte zurück und liefert die Kreuzgleichungskovarianz und die Restriktionstests, die ein Nachfrageanalyst benötigt.

## Datenquellen

| Datensatz | Zeilen | Granularität | Schlüsselvariablen | Beschreibung |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | eine Zeile pro monatlicher Marktbeobachtung | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | Synthetisches monatliches Panel des Wohnenergiemarkts. `lp_elec`/`lp_gas` sind die logarithmierten Realpreise von Strom und Erdgas; `lincome` ist das logarithmierte reale Haushaltseinkommen; `w_elec`/`w_gas` sind die Ausgaben-Budgetanteile für Strom und Erdgas, erzeugt aus einer bekannten Nachfragestruktur im AIDS-Stil plus korreliertem Kreuzgleichungsrauschen. |

# Schätzung eines Nachfragesystems für Haushaltsenergie mit scheinbar unverbundener Regression

Ein regionales Gas- und Stromversorgungsunternehmen möchte verstehen, wie Haushalte ihr Energiebudget zwischen **Strom** und **Erdgas** umverteilen, wenn sich relative Preise und Einkommen ändern. Der natürliche Rahmen ist ein **Nachfragesystem**: eine Menge gemeinsam geschätzter Budgetanteilsgleichungen.

Zwei Merkmale machen die gemeinsame Schätzung zum richtigen Werkzeug:

- Die Anteilsgleichungen schöpfen aus einem gemeinsamen Haushaltsbudget, sodass ihre Fehler **kreuzkorreliert** sind — wenn ein Haushalt mehr für Strom ausgibt, gibt er weniger für Gas aus. Die gemeinsame Schätzung der Gleichungen mit **scheinbar unverbundener Regression (SUR)** nutzt diese Korrelation zur Effizienzsteigerung.
- Die Wirtschaftstheorie legt **lineare Restriktionen** auf (Aufsummierung, Homogenität, Symmetrie), die ein Systemschätzer direkt durchsetzen kann.

`PROC SYSLIN` mit der `SUR`-Option ist genau für diese Situation gemacht. Es passt jede Anteilsgleichung an, schätzt die Kreuzgleichungs-Fehlerkovarianz und passt das System mit dieser Kovarianz erneut an, um effiziente, theoriekohärente Elastizitäten zu liefern — zusammen mit der Cross-Model-Kovarianzmatrix und gemeinsamen Restriktionstests.

In diesem Notebook:

1. Erzeugen wir ein synthetisches monatliches Energiemarkt-Panel aus einer bekannten Anteilsstruktur.
2. Passen wir ein Anteilssystem mit zwei Gleichungen per SUR an.
3. Vergleichen wir die gemeinsame SUR-Anpassung mit gleichungsweiser OLS.
4. Legen wir eine Homogenitätsrestriktion auf und lesen ihren gemeinsamen F-Test ab.
5. Extrahieren wir die Koeffiziententabelle für die Elastizitätsberichterstattung.

## Schritt 1 — Ein synthetisches Energiemarkt-Panel erzeugen

Wir simulieren 60 monatliche Beobachtungen eines kleinen **Energienachfragesystems mit zwei Gütern** (Strom und Erdgas). Der datenerzeugende Prozess ist ein linearisiertes Budgetanteilsmodell im AIDS-Stil:

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

Wir bauen absichtlich eine **Kreuzgleichungskorrelation** ein: Wenn Haushalte mehr für Strom ausgeben, geben sie weniger für Gas aus, sodass `e1` und `e2` negativ korreliert sind. Ein gemeinsamer Energiemarkt-Preisfaktor treibt zudem beide Log-Preise gemeinsam. Das sind die Merkmale, die die gemeinsame SUR-Schätzung gegenüber der Einzelanpassung jeder Gleichung belohnen.

In [1]:
DATEN energy;
   AUFRUFEN streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   AUSFÜHRUNG month = 1 BIS 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      AUSGABE;
   ENDE;

   BEHALTEN month lp_elec lp_gas lincome w_elec w_gas;
AUSFÜHREN;

PROZEDUR MITTELWERTE DATEN=energy n mean std MIN MAX maxdec=3;
   VAR w_elec w_gas lp_elec lp_gas lincome;
   BEZEICHNUNG w_elec="Ausgabenanteil Strom" w_gas="Ausgabenanteil Gas"
         lp_elec="Log-Preis Strom" lp_gas="Log-Preis Gas"
         lincome="Log-Einkommen";
AUSFÜHREN;

                                                  The MEANS Procedure

 Variable  Label                       N           Mean     Std Dev     Minimum     Maximum
 ------------------------------------------------------------------------------------------
 w_elec    Ausgabenanteil Strom       60          0.440       0.011       0.417       0.464
 w_gas     Ausgabenanteil Gas         60          0.228       0.014       0.198       0.256
 lp_elec   Log-Preis Strom            60          4.617       0.142       4.354       4.911
 lp_gas    Log-Preis Gas              60          4.211       0.162       3.818       4.566
 lincome   Log-Einkommen              60         10.916       0.088      10.723      11.174
 ------------------------------------------------------------------------------------------




NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Schritt 2 — SUR-Basisschätzung des Nachfragesystems

Wir schätzen beide Anteilsgleichungen gemeinsam. Die `SUR`-Option weist `PROC SYSLIN` an, die Kreuzgleichungs-Fehlerkovarianz zu schätzen und für eine effiziente Feasible-GLS-Anpassung zu verwenden. Zwei benannte `MODEL`-Anweisungen (`elec` und `gas`) definieren das System; jede regressiert einen Budgetanteil auf die beiden Log-Preise und das Log-Einkommen.

SYSLIN berichtet die Parameterschätzungen jeder Gleichung und am Ende die **Cross-Model Covariance Matrix** — die geschätzte Kovarianz der Fehler beider Gleichungen, die SUR ausnutzt.

In [2]:
PROZEDUR syslin DATEN=energy sur;
   strom: MODELL w_elec = lp_elec lp_gas lincome;
   gas:   MODELL w_gas  = lp_elec lp_gas lincome;
AUSFÜHREN;


                       The SYSLIN Procedure

                  SUR Estimation

  Model strom Dependent Variable: w_elec

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: w_gas

  Number of Observations                       60
  SSE                                      0


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Schritt 3 — Vergleich mit gleichungsweiser OLS

Um zu sehen, was SUR uns bringt, passen wir dieselben zwei Gleichungen erneut mit der Methode der kleinsten Quadrate an (der Standardmethode, jeweils eine Gleichung nach der anderen). OLS ignoriert die Kreuzgleichungs-Fehlerkorrelation und ist daher konsistent, aber weniger effizient als SUR, wenn diese Korrelation von null verschieden ist — wie es hier per Konstruktion der Fall ist.

Der Vergleich der beiden Koeffiziententabellen zeigt, wie sich die Schätzungen und ihre Standardfehler verschieben, sobald die Systemstruktur berücksichtigt wird.

In [3]:
PROZEDUR syslin DATEN=energy ols;
   strom: MODELL w_elec = lp_elec lp_gas lincome;
   gas:   MODELL w_gas  = lp_elec lp_gas lincome;
AUSFÜHREN;


                       The SYSLIN Procedure

                  OLS Estimation

  Model strom Dependent Variable: w_elec

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: w_gas

  Number of Observations                       60
  SSE                                      0


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## Schritt 4 — Wirtschaftstheorie auferlegen und testen

Die Nachfragetheorie besagt, dass nominale Preis-/Einkommenseffekte der **Homogenität vom Grad null** genügen sollten: Eine Skalierung aller Preise und des Einkommens lässt die reale Nachfrage unverändert, sodass sich die Preis- und Einkommenskoeffizienten innerhalb einer Gleichung zu null summieren. Wir legen dies dem Stromanteil mit einer `RESTRICT`-Anweisung auf.

SYSLIN passt das System unter der Nebenbedingung erneut an und berichtet einen F-Test der **Restriction Results**, ob die Restriktion mit den Daten vereinbar ist — ein direkter, gemeinsamer Test der Homogenitätshypothese.

In [4]:
PROZEDUR syslin DATEN=energy sur;
   strom: MODELL w_elec = lp_elec lp_gas lincome;
   gas:   MODELL w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
AUSFÜHREN;


                       The SYSLIN Procedure

                  SUR Estimation

  Model strom Dependent Variable: w_elec

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: w_gas

  Number of Observations                       60
  SSE                                      0


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Schritt 5 — Schätzungen für die Elastizitätsberichterstattung erfassen

Für einen Abschlussbericht speichern wir die konvergierten Koeffizienten mit `OUTEST=`. Der resultierende Datensatz enthält eine Zeile pro Gleichung mit den Achsenabschnitts- und Steigungsschätzungen plus Anpassungsstatistiken, die in die Eigen- und Kreuzpreiselastizitätsberechnungen des Versorgungsunternehmens bei Stichprobenmittel-Anteilen einfließen. Wir drucken ihn zur Dokumentation.

In [5]:
PROZEDUR syslin DATEN=energy sur outest=demand_est;
   strom: MODELL w_elec = lp_elec lp_gas lincome;
   gas:   MODELL w_gas  = lp_elec lp_gas lincome;
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=demand_est noobs;
   TITEL "SUR-Nachfragesystem: geschätzte Koeffizienten";
AUSFÜHREN;
TITEL;


                       The SYSLIN Procedure

                  SUR Estimation

  Model strom Dependent Variable: w_elec

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: w_gas

  Number of Observations                       60
  SSE                                      0


NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## Interpretation der Ergebnisse

**Vorzeichenkohärenz.** Die SUR-Schätzungen geben die in die Daten eingebaute Substitutionsstruktur wieder. In der Stromanteilsgleichung ist der **Eigen-Log-Preiskoeffizient positiv** (`LP_ELEC` = 0.112), während der **Kreuzpreiskoeffizient negativ** ist (`LP_GAS` = -0.098); in der Gasanteilsgleichung spiegelt sich das Muster (eigen `LP_GAS` = 0.114, kreuz `LP_ELEC` = -0.075). Jeder Eigen- und Kreuzpreiseffekt ist statistisch signifikant (jedes `Pr > |t|` unter 0.005), sodass die Substitutionsvorzeichen fest identifiziert sind und nicht Artefakte einer verrauschten Anpassung.

**SUR nutzt die Kreuzgleichungskorrelation.** Die **Cross-Model Covariance Matrix** zeigt eine negative Nebendiagonale (-0.000068): Haushalte tauschen Stromausgaben gegen Gasausgaben, genau wie konstruiert. Da diese Kovarianz von null verschieden ist, ist die gemeinsame SUR-Schätzung effizienter als die Einzelanpassung jeder Gleichung — die SUR-Standardfehler in Schritt 2 sind durchweg etwas kleiner als die gleichungsweisen OLS-Standardfehler in Schritt 3 (zum Beispiel sinkt der Standardfehler von Gas `LP_ELEC` von 0.0264 unter OLS auf 0.0255 unter SUR), während die Punktschätzungen unverändert bleiben. Diese engere Inferenz ist der Gewinn aus der Modellierung des Systems statt zweier getrennter Regressionen.

**Die Theorie hält stand.** Die Auferlegung der **Homogenität vom Grad null** auf den Stromanteil (Preis- und Einkommenskoeffizienten summieren sich zu null) via `RESTRICT` ergab einen Restriction-Results-F-Test von 2.14 mit `Pr > F` = 0.149. Die Restriktion wird **nicht abgelehnt**, sodass die Homogenitätsvorhersage der Konsumnachfragetheorie mit dieser Stichprobe vereinbar ist — das Versorgungsunternehmen kann die eingeschränkten, theoriekohärenten Schätzungen mit Zuversicht in die Berichterstattung übernehmen.

**Operativer Einsatz.** Die `OUTEST=`-Tabelle speichert eine Zeile pro Gleichung mit den Achsenabschnitts- und Steigungsschätzungen und Anpassungsstatistiken. Diese Koeffizienten lassen sich direkt in Eigen- und Kreuzpreiselastizitäten und eine Einkommens- (Ausgaben-)Elastizität bei den Stichprobenmittel-Anteilen umrechnen (`W_ELEC` ~ 0.44, `W_GAS` ~ 0.23 aus Schritt 1) und geben dem Versorgungsunternehmen eine belastbare, theoriekonsistente Grundlage für Nachfrageprognosen und Tarifgestaltungsszenarien.